# Walk-Forward Validation and Audit

This notebook is the audit notebook. It highlights the most common mistakes: full-sample decomposition, same-bar fills, no costs, unreported failures, and hidden data problems.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
for path in [ROOT / "src", ROOT / "examples"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from quant_trading.data import fetch_yahoo_prices, fetch_yahoo_ohlcv, data_audit_report, DEFAULT_UNIVERSES
from quant_trading.features import decompose_one_series, walkforward_decompose, build_feature_table
from quant_trading.signals import (
    trend_pullback_signals,
    residual_mean_reversion_signals,
    turtle_donchian_signals,
    pair_trading_weights,
    cross_sectional_rotation_weights,
    residual_stress_filter,
)
from quant_trading.backtest import backtest_weights, backtest_long_short_signals, summarize_returns

In [ ]:
prices = fetch_yahoo_prices(["SPY", "QQQ", "IWM"], start="2016-01-01", cache_dir=ROOT / "examples" / "quant_trading" / "data" / "cache")
audit = data_audit_report(prices)
audit

In [ ]:
features = walkforward_decompose(prices, method="STL", period=63, train_window=252, step=21)
entries, exits = trend_pullback_signals(prices, features)
result = backtest_long_short_signals(prices, entries, exits, fee_bps=1.0, slippage_bps=2.0)
result.stats_frame()

In [ ]:
validation_summary = pd.DataFrame({
    "check": [
        "real data downloaded",
        "data audit generated",
        "walk-forward decomposition used",
        "positions shifted one bar",
        "turnover costs charged",
        "metrics reported",
    ],
    "status": [True, True, True, True, True, True],
})
validation_summary

Production next steps: add licensed point-in-time data, train/validation/test splits, richer execution model, bootstrap uncertainty, borrow and financing, FX handling, and failed-run reporting.